In [ ]:
#1.4

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re


class POSTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dims, output_dim, context_size, num_features, activation='tanh'):
        super(POSTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        layers = []
        input_dim = embedding_dim * (2 * context_size + 1) + num_features

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            if activation == 'tanh':
                layers.append(nn.Tanh())
            elif activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'sigmoid':
                layers.append(nn.Sigmoid())
            input_dim = hidden_dim

        self.hidden_layers = nn.Sequential(*layers)
        self.output = nn.Linear(input_dim, output_dim)

    def forward(self, x, features):
        embedded = self.embedding(x)
        embedded = embedded.view(x.shape[0], -1)
        combined = torch.cat((embedded, features), dim=1)
        hidden = self.hidden_layers(combined)
        output = self.output(hidden)
        return output


def extract_features(word, prev_word, next_word):
    features = []
    features.append(int(word.istitle()))  # Is capitalized
    features.append(int(word.isupper()))  # Is all uppercase
    features.append(int(bool(re.search(r'\d', word))))  # Contains digit
    features.append(int('@' in word))  # Contains @
    features.append(int('#' in word))  # Contains #
    features.append(len(word))  # Word length
    features.append(int(word.startswith('un')))  # Starts with 'un'
    features.append(int(prev_word.istitle()))  # Previous word is capitalized
    features.append(int(next_word.istitle()))  # Next word is capitalized
    features.append(int(word.lower() in ['the', 'a', 'an']))  # Is article
    return features


class TwitterPOSDataset(Dataset):
    def __init__(self, file_path, word_to_idx, tag_to_idx, context_size):
        self.data = []
        self.word_to_idx = word_to_idx
        self.tag_to_idx = tag_to_idx
        self.context_size = context_size

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        tokens = ['<s>'] * context_size + ['<s>']
        tags = ['<s>'] * context_size + ['<s>']
        for line in lines:
            if line.strip():
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    word, tag = parts
                    tokens.append(word)
                    tags.append(tag)
            else:
                tokens.extend(['</s>'] * (context_size + 1))
                tags.extend(['</s>'] * (context_size + 1))
                tokens.extend(['<s>'] * (context_size + 1))
                tags.extend(['<s>'] * (context_size + 1))

        tokens.extend(['</s>'] * (context_size + 1))
        tags.extend(['</s>'] * (context_size + 1))

        for idx in range(context_size, len(tokens) - context_size - 1):
            context = [self.word_to_idx.get(tokens[idx + i], self.word_to_idx['UUUNKKK']) for i in range(-context_size, context_size + 1)]
            target = self.tag_to_idx.get(tags[idx], self.tag_to_idx['X'])

            features = extract_features(tokens[idx], tokens[idx-1], tokens[idx+1])

            self.data.append((context, target, features))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0]), self.data[idx][1], torch.tensor(self.data[idx][2], dtype=torch.float)


def train_model(model, train_loader, dev_loader, criterion, optimizer, epochs):
    best_dev_accuracy = 0
    for epoch in range(epochs):
        model.train()
        for batch_contexts, batch_tags, batch_features in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_contexts, batch_features)
            loss = criterion(outputs, batch_tags)
            loss.backward()
            optimizer.step()

        dev_accuracy = evaluate_model(model, dev_loader)
        print(f'Epoch {epoch+1}, Dev Accuracy: {dev_accuracy:.2f}%')

        if dev_accuracy > best_dev_accuracy:
            best_dev_accuracy = dev_accuracy
            torch.save(model.state_dict(), 'best_model.pth')

    print(f'Best Dev Accuracy: {best_dev_accuracy:.2f}%')


def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_contexts, batch_tags, batch_features in data_loader:
            outputs = model(batch_contexts, batch_features)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_tags.size(0)
            correct += (predicted == batch_tags).sum().item()
    return 100 * correct / total


def main():
    EMBEDDING_DIM = 50
    BATCH_SIZE = 16
    EPOCHS = 5
    LEARNING_RATE = 0.02
    NUM_FEATURES = 10

    word_to_idx = {}
    embeddings = []
    with open('twitter-embeddings.txt', 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            parts = line.strip().split()
            word = parts[0]
            embedding = [float(val) for val in parts[1:]]
            word_to_idx[word] = idx
            embeddings.append(embedding)

    for token in ['UUUNKKK', '<s>']:
        if token not in word_to_idx:
            word_to_idx[token] = len(word_to_idx)
            embeddings.append([0.0] * EMBEDDING_DIM)  # Initialize with zeros

    word_to_idx['<s>'] = word_to_idx['</s>']

    tag_to_idx = {
        'N': 0, 'O': 1, 'S': 2, 'L': 3, '^': 4, 'Z': 5, 'M': 6, 'V': 7, 'A': 8, 'R': 9,
        '!': 10, 'D': 11, 'P': 12, '&': 13, 'T': 14, 'X': 15, 'Y': 16, '#': 17, '@': 18,
        '~': 19, 'U': 20, 'E': 21, '$': 22, ',': 23, 'G': 24, '<s>': 25, '</s>': 26
    }

    configurations = [
        {'hidden_dims': [], 'activation': 'tanh', 'context_size': w}
        for w in [0, 1, 2]
    ] + [
        {'hidden_dims': [dim], 'activation': act, 'context_size': 1}
        for dim in [256, 512]
        for act in ['tanh', 'relu', 'sigmoid']
    ] + [
        {'hidden_dims': [dim1, dim2], 'activation': 'tanh', 'context_size': 1}
        for dim1, dim2 in [(256, 256), (512, 512)]
    ]

    for config in configurations:
        print(f"Training with configuration: {config}")

        train_dataset = TwitterPOSDataset('twpos-train.tsv', word_to_idx, tag_to_idx, config['context_size'])
        dev_dataset = TwitterPOSDataset('twpos-dev.tsv', word_to_idx, tag_to_idx, config['context_size'])
        devtest_dataset = TwitterPOSDataset('twpos-devtest.tsv', word_to_idx, tag_to_idx, config['context_size'])

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
        devtest_loader = DataLoader(devtest_dataset, batch_size=BATCH_SIZE)

        model = POSTagger(len(word_to_idx), EMBEDDING_DIM, config['hidden_dims'],
                          len(tag_to_idx), config['context_size'], NUM_FEATURES,
                          activation=config['activation'])

        model.embedding.weight.data.copy_(torch.tensor(embeddings))

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

        train_model(model, train_loader, dev_loader, criterion, optimizer, EPOCHS)

        model.load_state_dict(torch.load('best_model.pth'))
        devtest_accuracy = evaluate_model(model, devtest_loader)
        print(f'Configuration: {config}, DevTest Accuracy: {devtest_accuracy:.2f}%')
        print('-' * 50)

if __name__ == '__main__':
    main()


Training with configuration: {'hidden_dims': [], 'activation': 'tanh', 'context_size': 0}
Epoch 1, Dev Accuracy: 59.06%
Epoch 2, Dev Accuracy: 70.80%
Epoch 3, Dev Accuracy: 76.11%
Epoch 4, Dev Accuracy: 78.47%
Epoch 5, Dev Accuracy: 80.19%
Best Dev Accuracy: 80.19%


<ipython-input-6-d97f08bce641>:194: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Configuration: {'hidden_dims': [], 'activation': 'tanh', 'context_size': 0}, DevTest Accuracy: 80.20%
--------------------------------------------------
Training with configuration: {'hidden_dims': [], 'activation': 'tanh', 'context_size': 1}
Epoch 1, Dev Accuracy: 66.79%
Epoch 2, Dev Accuracy: 76.61%
Epoch 3, Dev Accuracy: 79.87%
Epoch 4, Dev Accuracy: 82.71%
Epoch 5, Dev Accuracy: 83.85%
Best Dev Accuracy: 83.85%
Configuration: {'hidden_dims': [], 'activation': 'tanh', 'context_size': 1}, DevTest Accuracy: 84.52%
--------------------------------------------------
Training with configuration: {'hidden_dims': [], 'activation': 'tanh', 'context_size': 2}
Epoch 1, Dev Accuracy: 74.75%
Epoch 2, Dev Accuracy: 79.48%
Epoch 3, Dev Accuracy: 82.62%
Epoch 4, Dev Accuracy: 84.18%
Epoch 5, Dev Accuracy: 85.55%
Best Dev Accuracy: 85.55%
Configuration: {'hidden_dims': [], 'activation': 'tanh', 'context_size': 2}, DevTest Accuracy: 85.61%
--------------------------------------------------
Training 